# 数据准备

In [1]:
import numpy as np
import pandas as pd

# 全局浮点数保留两位小数
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# 原始订单数据
orders = pd.DataFrame({
    'order_id': [f'O{number}' for number in range(1001, 1019)],
    'region': ['华东','华北','华南','华东','西南','华北','华南','华东','西南','华北','华东','华南','西南','华东','华北','华南','华东','西南'],
    'product': ['机械键盘','无线鼠标','显示器','扩展坞','机械键盘','显示器','无线鼠标','显示器','扩展坞','机械键盘','无线鼠标','扩展坞','显示器','机械键盘','扩展坞','显示器','无线鼠标','机械键盘'],
    'category': ['外设','外设','显示设备','配件','外设','显示设备','外设','显示设备','配件','外设','外设','配件','显示设备','外设','配件','显示设备','外设','外设'],
    'quantity': [2,3,1,4,5,2,6,1,3,2,8,2,1,3,5,2,4,6],
    'unit_price': [289,129,1299,399,289,1299,129,1299,399,289,129,399,1299,289,399,1299,129,289],
    'member_level': ['金卡','普通','银卡','金卡','银卡','普通','金卡','银卡','普通','金卡','银卡','金卡','普通','银卡','金卡','金卡','普通','银卡'],
    'coupon_rate': [0.05,0.00,0.08,0.10,0.05,0.00,0.12,0.05,0.00,0.08,0.10,0.05,0.00,0.12,0.05,0.08,0.00,0.10],
    'salesperson': ['小林','小周','小陈','小林','小赵','小周','小陈','小林','小赵','小周','小林','小陈','小赵','小林','小周','小陈','小林','小赵']
})

## 任务 1：快速理解数据

In [2]:
row_num, col_num = orders.shape
print(f"数据行数：{row_num}，列数：{col_num}")
print("所有列名：", orders.columns.tolist())

ser_region = orders['region']
df_three = orders[['order_id', 'product', 'quantity']]
print("region单列类型：", type(ser_region))
print("三列数据表类型：", type(df_three))

slice_iloc = orders.iloc[3:8, 0:4]
print(slice_iloc)

east_loc = orders.loc[orders['region'] == '华东', ['order_id', 'product', 'member_level']]
print(east_loc)

数据行数：18，列数：9
所有列名： ['order_id', 'region', 'product', 'category', 'quantity', 'unit_price', 'member_level', 'coupon_rate', 'salesperson']
region单列类型： <class 'pandas.Series'>
三列数据表类型： <class 'pandas.DataFrame'>
  order_id region product category
3    O1004     华东     扩展坞       配件
4    O1005     西南    机械键盘       外设
5    O1006     华北     显示器     显示设备
6    O1007     华南    无线鼠标       外设
7    O1008     华东     显示器     显示设备
   order_id product member_level
0     O1001    机械键盘           金卡
3     O1004     扩展坞           金卡
7     O1008     显示器           银卡
10    O1011    无线鼠标           银卡
13    O1014    机械键盘           银卡
16    O1017    无线鼠标           普通


loc 更适合长期业务代码的原因
loc 基于行列标签名取值，不受行索引顺序变动、数据插入删除影响；iloc 是纯位置下标，数据增减后下标对应行会错乱，业务数据表常更新，因此推荐 loc。

## 任务 2：构造订单结算指标

In [3]:
analysis = orders.assign(
    # 商品总金额 = 数量*单价
    gross_amount = lambda x: (x['quantity'] * x['unit_price']).round(2),
    # 会员折扣：金卡0.1，银卡0.05，普通0
    member_discount = lambda x: np.where(x['member_level'] == '金卡', 0.10,
                                np.where(x['member_level'] == '银卡', 0.05, 0.00)),
    # 优惠后应付金额
    payable_amount = lambda x: (x['gross_amount'] * (1 - x['member_discount']) * (1 - x['coupon_rate'])).round(2),
    # 运费：应付≥1000免运费，否则20
    shipping_fee = lambda x: np.where(x['payable_amount'] >= 1000, 0, 20),
    # 最终实付金额
    final_amount = lambda x: (x['payable_amount'] + x['shipping_fee']).round(2)
)

# 展示前8行核心结算字段
show_cols = ['order_id','gross_amount','member_discount','payable_amount','shipping_fee','final_amount']
print(analysis[show_cols].head(8))

  order_id  gross_amount  member_discount  payable_amount  shipping_fee  \
0    O1001           578             0.10          494.19            20   
1    O1002           387             0.00          387.00            20   
2    O1003          1299             0.05         1135.33             0   
3    O1004          1596             0.10         1292.76             0   
4    O1005          1445             0.05         1304.11             0   
5    O1006          2598             0.00         2598.00             0   
6    O1007           774             0.10          613.01            20   
7    O1008          1299             0.05         1172.35             0   

   final_amount  
0        514.19  
1        407.00  
2       1135.33  
3       1292.76  
4       1304.11  
5       2598.00  
6        633.01  
7       1172.35  


## 任务 3：复杂条件筛选

In [4]:
# 条件1：地区华东 或 华南
cond1 = analysis['region'].isin(['华东', '华南'])
# 条件2：最终金额≥700
cond2 = analysis['final_amount'] >= 700
# 条件3：数量≥2 或者 会员是金卡
cond3 = (analysis['quantity'] >= 2) | (analysis['member_level'] == '金卡')

# 组合总筛选掩码：条件1 & 条件2 & 条件3
mask = cond1 & cond2 & cond3

# 筛选、指定列、金额降序
focus_order = analysis.loc[mask, ['order_id','region','product','quantity','member_level','final_amount']]
focus_order = focus_order.sort_values('final_amount', ascending=False)
print(focus_order)

   order_id region product  quantity member_level  final_amount
15    O1016     华南     显示器         2           金卡       2151.14
3     O1004     华东     扩展坞         4           金卡       1292.76
10    O1011     华东    无线鼠标         8           银卡        902.36
13    O1014     华东    机械键盘         3           银卡        744.81
11    O1012     华南     扩展坞         2           金卡        702.29


&、| 两侧括号原因
pandas 中&/|运算符优先级高于比较运算符>= ==，不加括号会先执行位运算再做比较，逻辑条件错乱，因此每个独立布尔表达式必须包裹括号。

## 任务 4：封装可复用处理函数

In [5]:
def add_order_level(df):
    # 拷贝避免修改原表
    df_copy = df.copy()
    # 嵌套np.where划分订单等级
    df_copy['order_level'] = np.where(
        df_copy['final_amount'] >= 2000, '战略订单',
        np.where(df_copy['final_amount'] >= 1000, '重点订单', '普通订单')
    )
    return df_copy

# pipe调用函数
leveled_orders = analysis.pipe(add_order_level)

# 统计各等级订单数量
level_count = leveled_orders['order_level'].value_counts()
print("各等级订单数量：")
print(level_count)

各等级订单数量：
order_level
普通订单    8
重点订单    8
战略订单    2
Name: count, dtype: int64


## 任务 5：一条链完成经营汇总

In [6]:
region_report = (
    analysis
    .pipe(add_order_level)          # 步骤1：新增订单等级
    .query('final_amount >= 500')    # 步骤2：过滤实付500以上订单
    .groupby(['region', 'order_level']) # 步骤3：地区+等级分组
    .agg(
        order_count = ('order_id', 'count'),
        quantity_sum = ('quantity', 'sum'),
        revenue_sum = ('final_amount', 'sum'),
        revenue_mean = ('final_amount', 'mean')
    ) # 步骤4：聚合指标
    .sort_values('revenue_sum', ascending=False) # 步骤5：总营收降序
    .round(2)
)
print(region_report)

                    order_count  quantity_sum  revenue_sum  revenue_mean
region order_level                                                      
西南     重点订单                   4            15      5282.68       1320.67
华东     普通订单                   4            17      2697.36        674.34
华北     战略订单                   1             2      2598.00       2598.00
华东     重点订单                   2             5      2465.11       1232.55
华南     战略订单                   1             2      2151.14       2151.14
华北     重点订单                   1             5      1705.72       1705.72
华南     普通订单                   2             8      1335.30        667.65
       重点订单                   1             1      1135.33       1135.33


## 任务 6：经营诊断与表达

In [7]:
# 1. 销售人员总成交金额，取最高销售
sales_total = leveled_orders.groupby('salesperson')['final_amount'].sum()
top_sales = sales_total.idxmax()
top_sales_total = sales_total.max()

# 2. 该销售各地区成交金额，提取最高地区
sales_region = leveled_orders[leveled_orders['salesperson'] == top_sales].groupby('region')['final_amount'].sum()
top_region = sales_region.idxmax()
top_region_revenue = sales_region.max()

# 3. 核心地区金额占此人总金额比例
rate = (top_region_revenue / top_sales_total).round(4)

# 输出结果
print(f"销冠销售：{top_sales}")
print(f"销售总成交金额：{top_sales_total:.2f}")
print(f"核心贡献地区：{top_region}")
print(f"核心地区成交金额：{top_region_revenue:.2f}")
print(f"核心地区营收贡献率：{rate*100:.2f}%")

# 业务结论
print(f"\n业务结论：销冠{top_sales}业绩高度依赖{top_region}区域，该地区贡献其全部营收的{rate*100:.2f}%，需巩固该区域渠道同时拓展其他区域分散营收。")

销冠销售：小赵
销售总成交金额：5282.68
核心贡献地区：西南
核心地区成交金额：5282.68
核心地区营收贡献率：100.00%

业务结论：销冠小赵业绩高度依赖西南区域，该地区贡献其全部营收的100.00%，需巩固该区域渠道同时拓展其他区域分散营收。
